In [12]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
"""
Fluence data analysis script

This script loads fluence and temperature data for further analysis.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from importlib import reload

import fluence_lib
from fluence_lib import (
    load_avg_fluence,
    load_temperature_data,
    prepare_sensor_data,
    simulate_current_sensor,
    plot_leakage_current
)

# Global constants
KB = 8.617333262145e-5  # Boltzmann constant in eV/K
T_REF = 293.15  # Reference temperature in K (20°C)
E_I_STAR = 1.0  # Activation energy in eV

# Default parameters
ALPHA_I = 1.23e-17
ALPHA_0_STAR = 7.07e-17
BETA = 3.29e-18
E_I = 1.30
K_0I = 1.2e13
T_0 = 1.0 #min
V=1

class Parameters:
    """
    Class to store physical parameters used in calculations.
    """
    def __init__(self):
        # Set initial values from global constants
        self.E_I_star = E_I_STAR
        self.kb = KB  # Add Boltzmann constant
        self.T_ref = T_REF
        self.k_0I = K_0I
        self.E_I = E_I
        self.alpha_I = ALPHA_I
        self.alpha_0_star = ALPHA_0_STAR
        self.beta = BETA
        self.t_0 = T_0
        self.V = V

    def get_kb(self) -> float:
        """Get Boltzmann constant in eV/K"""
        return self.kb

    def get_E_I_star(self) -> float:
        """Get activation energy in eV"""
        return self.E_I_star
    def set_E_I_star(self, value: float):
        self.E_I_star = value
        
    def set_T_ref(self, value: float):
        self.T_ref = value
        
    def set_k_0I(self, value: float):
        self.k_0I = value
        
    def set_E_I(self, value: float):
        self.E_I = value
        
    def set_alpha_I(self, value: float):
        self.alpha_I = value
        
    def set_alpha_0_star(self, value: float):
        self.alpha_0_star = value
        
    def set_beta(self, value: float):
        self.beta = value
        
    def set_t_0(self, value: float):
        self.t_0 = value
    
    def set_V(self, value: float):
        self.V = value


In [17]:
def load_data():
    """
    Load both fluence and temperature data.
    """
    fluence_file = os.path.join('in_data', 'avgFluenceBinned.csv')
    temp_file = os.path.join('in_data', 'run2_temp_A_60s.csv')
    
    fluence_df = load_avg_fluence(fluence_file)
    print("Fluence data loaded successfully!")
    
    temp_df = load_temperature_data(temp_file)
    print("Temperature data loaded successfully!")
    
    return fluence_df, temp_df

def test_single_sensor(fluence_df: pd.DataFrame, temp_df: pd.DataFrame, params):
    """
    Test current simulation for a single sensor.
    """
    sensor_name = 'VL01_CT'  # Example sensor
    
    # Prepare data for this sensor
    fdf,tdf = fluence_lib.prepare_sensor_data(fluence_df, temp_df, sensor_name)

    current_df = []
    # Simulate current for this sensor
    current_df = fluence_lib.simulate_current_sensor(fdf,tdf, sensor_name, params)
    
    #print(f"\nSimulated current for sensor {sensor_name}")
    #print(current_df.head())
    
    # Plot the results
    #output_path = os.path.join('out_data', f'leakage_current_{sensor_name}.png')
    #os.makedirs('out_data', exist_ok=True)
    #plot_leakage_current(current_df, sensor_name, output_path)
    #print(f"\nPlot saved to {output_path}")
    return current_df
    #return current_df

In [24]:

import fluence_lib

reload(fluence_lib)
params = Parameters()
    
# Load data
fluence_df, temp_df = load_data()
    
# Test for a single sensor
df = test_single_sensor(fluence_df, temp_df, params)

Fluence data loaded successfully!


/home/maria/magisterka/hamburg_new/fluence_lib.py:73: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  fluence_series = fluence_series.reindex(pd.date_range(fluence_series.index.min(), fluence_series.index.max(), freq='10T'), fill_value=0)


Temperature data loaded successfully!
Starting current simulation...
Calculating time factors...
Starting main calculation loop...


  0%|          | 468/131953 [00:00<00:28, 4675.26it/s]

Progress: 0.0%, Current sum_1: 4.61e-06, sum_2: 3.56e+04


 10%|█         | 13854/131953 [00:02<00:23, 4952.40it/s]

... (15 outputs left out)

 91%|█████████ | 119778/131953 [00:20<00:01, 7570.97it/s]

Progress: 90.0%, Current sum_1: 2.36e-06, sum_2: 1.14e+04


100%|██████████| 131953/131953 [00:22<00:00, 5892.37it/s]

Progress: 100.0%, Current sum_1: 2.68e-11, sum_2: 5.62e-01
Simulation completed!


In [25]:
df

,current,fluence,temperature
ts,,,
2016-04-23 00:50:00,2.460180e-10,5.070752e+06,243.567500
2016-04-23 01:00:00,2.594215e-10,5.347014e+06,243.567500
2016-04-23 01:10:00,2.594216e-10,5.347014e+06,243.567500
2016-04-23 01:20:00,2.594217e-10,5.347014e+06,243.567500
2016-04-23 01:30:00,2.594218e-10,5.347014e+06,243.567500
...,...,...,...
2018-10-26 08:10:00,1.944414e-09,2.336469e+07,243.469488
2018-10-26 08:20:00,3.206789e-09,3.819840e+07,243.496523
2018-10-26 08:30:00,3.242943e-09,3.819840e+07,243.496523


In [ ]:
output_path = os.path.join('out_data', f'leakage_current_{'VL01_CT'}.png')
fluence_lib.plot_leakage_current(df, 'VL01_CT', output_path)